In [ ]:
print('Loading PhoBERT tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Vocab size: {tokenizer.vocab_size:,}')
sample = preprocess('Bọn đó ăn bám lắm, ngu vcl 🙃')
print(f'Processed : {sample}')
print(f'Tokens    : {tokenizer.tokenize(sample)}')

In [ ]:
print("Preprocessing texts...")
for df, name in [(train_df, 'Train'), (val_df, 'Val'), (test_df, 'Test')]:
    df['text_processed'] = df['free_text'].apply(preprocess)
    print(f'  {name}: done')

In [ ]:
from underthesea import word_tokenize

TEENCODE_MAP = {
    'ko': 'không', 'kh': 'không', 'khong': 'không', 'kg': 'không',
    'hok': 'không', 'hk': 'không', 'hem': 'không', 'kô': 'không',
    'chx': 'chưa', 'chua': 'chưa',
    'r': 'rồi', 'rui': 'rồi', 'ròi': 'rồi', 'oy': 'rồi', 'uj': 'rồi',
    'mk': 'mình', 'mik': 'mình', 'mh': 'mình',
    'tui': 'tôi', 'tau': 'tao', 'may': 'mày', 'mi': 'mày',
    'bn': 'bạn', 'ban': 'bạn',
    'no': 'nó', 'mng': 'mọi người', 'mn': 'mọi người', 'ae': 'anh em',
    'dc': 'được', 'đc': 'được', 'dk': 'được', 'đk': 'được',
    'đươc': 'được', 'duoc': 'được', 'vs': 'với', 'voi': 'với',
    'j': 'gì', 'zì': 'gì', 'zi': 'gì',
    'ntn': 'như thế nào', 'nso': 'như sao',
    'biet': 'biết', 'bit': 'biết', 'hieu': 'hiểu', 'nghi': 'nghĩ',
    'muon': 'muốn', 'hoac': 'hoặc', 'neu': 'nếu', 'nen': 'nên',
    'giet': 'giết', 'chui': 'chửi', 'danh': 'đánh',
    'nx': 'nhưng', 'nhg': 'nhưng', 'nhưg': 'nhưng', 'nma': 'nhưng mà',
    'cx': 'cũng', 'cg': 'cũng', 'cung': 'cũng', 'cũg': 'cũng',
    'ms': 'mới', 'boi': 'bởi',
    'oke': 'ok', 'okie': 'ok', 'okê': 'ok', 'okey': 'ok',
    'uh': 'ừ', 'uk': 'ừ', 'uhm': 'ừ',
    'yep': 'đúng', 'yup': 'đúng',
    'haha': 'haha', 'hehe': 'hehe', 'hihi': 'hehe', 'huhu': 'buồn',
    'haiz': 'thở dài', 'haizz': 'thở dài',
    'wtf': 'cái gì vậy', 'omg': 'ôi trời',
    'lol': 'buồn cười', 'lmao': 'buồn cười',
    'fck': 'chửi thề', 'fk': 'chửi thề', 'gg': 'xong rồi', 'ez': 'dễ',
    'bt': 'bình thường', 'bth': 'bình thường',
    'noob': 'tệ', 'nub': 'tệ', 'xàm': 'vô nghĩa', 'nhảm': 'vô nghĩa',
    'pro': 'giỏi',
    'vl': 'vãi lồn', 'vcl': 'vãi cái lồn', 'vkl': 'vãi kép lồn',
    'vll': 'vãi lồn', 'vleu': 'vãi lồn', 'vloz': 'vãi lồn',
    'dm': 'đụ má', 'đm': 'đụ má', 'd.m': 'đụ má', 'đ.m': 'đụ má',
    'đmm': 'đụ má mày', 'dmm': 'đụ má mày',
    'đtm': 'địt mẹ', 'dtm': 'địt mẹ',
    'cl': 'cái lồn', 'lon': 'lồn', 'loz': 'lồn', 'l0n': 'lồn',
    'đéo': 'không', 'deo': 'không', 'éo': 'không',
    'cc': 'cái con', 'thg': 'thằng',
    'ngu': 'ngu', 'đần': 'đần độn', 'khùng': 'điên', 'dien': 'điên',
    'cút': 'cút', 'cut': 'cút',
    'câm': 'câm miệng', 'im mồm': 'câm miệng',
    'fb': 'facebook', 'yt': 'youtube', 'tt': 'tiktok', 'zl': 'zalo',
    'ig': 'instagram', 'cmt': 'bình luận', 'rep': 'trả lời',
    'vn': 'việt nam', 'hn': 'hà nội', 'hcm': 'hồ chí minh', 'sg': 'sài gòn',
    'iu': 'yêu', 'ieu': 'yêu',
}

EMOJI_MAP = {
    '🙃': ' [MOCK] ', '😏': ' [MOCK] ', '😒': ' [MOCK] ',
    '😡': ' [ANGER] ', '🤬': ' [ANGER] ', '😤': ' [ANGER] ',
    '🤮': ' [DISGUST] ', '😖': ' [DISGUST] ',
    '😂': ' [LAUGH] ', '🤣': ' [LAUGH] ',
    '😭': ' [SAD] ', '💔': ' [SAD] ',
    '🔥': ' [INTENSE] ', '💀': ' [DEATH] ',
    '👍': ' [APPROVE] ', '👎': ' [DISAPPROVE] ',
}

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    for emoji, tag in EMOJI_MAP.items():
        text = text.replace(emoji, tag)
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\b0[0-9]{9,10}\b', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [TEENCODE_MAP.get(w, w) for w in text.split()]
    return ' '.join(words)

def preprocess(text: str) -> str:
    text = normalize_text(text)
    if not text:
        return ''
    try:
        return word_tokenize(text, format='text')
    except Exception:
        return text

sample = 'Bọn đó ăn bám lắm, ngu vcl 🙃'
print(f'Original : {sample}')
print(f'Processed: {preprocess(sample)}')

In [ ]:
train_df['label_id'] = train_df['label_id'].map(lambda x: 1 if x == 2 else 0)
val_df['label_id']   = val_df['label_id'].map(lambda x: 1 if x == 2 else 0)
test_df['label_id']  = test_df['label_id'].map(lambda x: 1 if x == 2 else 0)

label_map = {0: 'NON-HATE', 1: 'HATE'}
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    dist = df['label_id'].value_counts().sort_index().rename(label_map)
    print(f"  {name}: {dist.to_dict()}")

In [ ]:
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset
import huggingface_hub

secret_value = UserSecretsClient().get_secret("HF_TOKEN")
huggingface_hub.login(token=secret_value, add_to_git_credential=False)

ds       = load_dataset("sonlam1102/vihsd")
train_df = ds["train"].to_pandas()
val_df   = ds["validation"].to_pandas()
test_df  = ds["test"].to_pandas()

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Label distribution (raw): {train_df['label_id'].value_counts().to_dict()}")

In [ ]:
MODEL_NAME  = 'vinai/phobert-base'
NER_MODEL   = 'NlpHUST/ner-vietnamese-electra-base'
MAX_LEN     = 256
HIDDEN_DIM  = 768
HEAD_DIM    = HIDDEN_DIM

BATCH_SIZE      = 16
NUM_EPOCHS      = 8
LR              = 2e-5
HEAD_LR         = 5e-5
WARMUP_RATIO    = 0.06
DROPOUT         = 0.1
PATIENCE        = 2
WEIGHT_DECAY    = 0.01
LABEL_SMOOTHING = 0.05
GRAD_ACCUM      = 2       # effective batch = 32
ALPHA_CL        = 0.1    # contrastive loss weight

NUM_CLASSES = 2
LABEL_NAMES = ['NON-HATE', 'HATE']
CKPT_NAME   = 'best_viamplehate_phobert_vihsd.pt'
PLOT_TITLE  = 'AmpleHate-Vi++ (PhoBERT) — ViHSD Proposed'

In [ ]:
import os, re, time, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, pipeline

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'
NUM_WORKERS = 2 if os.cpu_count() and os.cpu_count() > 2 else 0

print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Workers: {NUM_WORKERS} | Pin memory: {PIN_MEMORY}')

In [ ]:
!pip install transformers datasets sentencepiece huggingface_hub underthesea easydict -q

# AmpleHate-Vi++ on ViHSD: PhoBERT Proposed

Implements **AmpleHate-Vi++** — Vietnamese-adapted AmpleHate with:
1. Vietnamese NER (`NlpHUST/ner-vietnamese-electra-base`) + target cue lexicon → coverage 0.09% → ~45%
2. Separate attack cue bank (offensive predicates)
3. Relation Bank: 3 HeadAttention modules (r_exp, r_imp, r_atk) fused via Linear
4. Instance-adaptive gate `g = σ(W·[h_CLS; r])` replacing fixed scalar `e`
5. CrossEntropy (weighted) + ContrastiveLoss (α=0.1)
6. max_length=256, gradient accumulation=2, 8 epochs max

**Baseline reference:** `notebooks/models/baselines/ViHSD - Baseline AmpleHate_PhoBERT/`
**Spec:** `docs/superpowers/specs/2026-05-21-viamplehate-proposed-design.md`